In [6]:

import os
import sys
import pandas as pd
import cv2
import logging


import os
import sys
import cv2, csv
import numpy as np
import pandas as pd
import mediapipe as mp
import warnings, logging
from dataclasses import dataclass

warnings.filterwarnings('ignore')
logging.getLogger('mediapipe').setLevel(logging.ERROR)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Definimos las conexiones de MediaPipe
mp_holistic = mp.solutions.holistic
POSE_CONNECTIONS = mp_holistic.POSE_CONNECTIONS
HAND_CONNECTIONS = mp_holistic.HAND_CONNECTIONS

@dataclass
class Landmark:
    x: float
    y: float
    z: float = 0.0  # Añadido z ya que se usa en el CSV
    landmark_id: int = None

In [7]:
def load_pose_from_csv(csv_path):
    """
    Lee un archivo CSV y devuelve un diccionario con los landmarks de 'pose'.
    Ignora 'hands' y 'face'.
    """
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        logging.error(f"Error leyendo CSV {csv_path}: {e}")
        return {}

    landmarks = {"pose": []}
    
    # Filtramos por tipo 'POSE' (según save_landmarks_to_csv en utils.py)
    if 'tipo' not in df.columns:
        return {}

    pose_rows = df[df['tipo'] == 'POSE']
    
    for _, row in pose_rows.iterrows():
        lm = Landmark(
            x=float(row['x']),
            y=float(row['y']),
            z=float(row.get('z', 0.0)),
            landmark_id=int(row['landmark_id'])
        )
        landmarks["pose"].append(lm)
        
    return landmarks

In [ ]:
def plot_landmarks(landmarks, background_img=None, tam=100, is_normalized=True):

    # Usamos tam=100 por defecto para que la imagen final sea nítida
    if background_img is not None:
        img = background_img.copy()
    else:
        img = np.zeros((tam, tam, 3), dtype=np.uint8)
    h, w = img.shape[:2]

    def to_pixel(point):
        if is_normalized:
            # point.x está en rango 0-100
            return (int(point.x * w / 100), int(point.y * h / 100))
        return (int(point.x * w), int(point.y * h))

    # Dibujar conexiones

    lines = [("pose", POSE_CONNECTIONS), ("left_hand", HAND_CONNECTIONS), ("right_hand", HAND_CONNECTIONS)]

    for key, connections in lines:
        if key in landmarks:
            lms = landmarks[key]
            for start_idx, end_idx in connections:
                if start_idx < len(lms) and end_idx < len(lms):
                    cv2.line(img, to_pixel(lms[start_idx]), to_pixel(lms[end_idx]), (255, 255, 255), 1)

    # Dibujar puntos

    for p in landmarks.get("pose", []):
        cv2.circle(img, to_pixel(p), 2, (0, 0, 255), -1)

    for hand in ["left_hand", "right_hand"]:
        for p in landmarks.get(hand, []):
            cv2.circle(img, to_pixel(p), 2, (255, 0, 0), -1)
    return img


def normalize_landmarks(landmarks, tam_target=100, margin=5):
    all_points = []
    for key in landmarks:
        all_points.extend(landmarks[key])
    
    if not all_points: return landmarks

    min_x, max_x = min(lm.x for lm in all_points), max(lm.x for lm in all_points)
    min_y, max_y = min(lm.y for lm in all_points), max(lm.y for lm in all_points)

    width = (max_x - min_x) if max_x != min_x else 0.01
    height = (max_y - min_y) if max_y != min_y else 0.01

    available_space = tam_target - (2 * margin)
    scale = available_space / max(width, height)
    offset_x = margin + (available_space - width * scale) / 2
    offset_y = margin + (available_space - height * scale) / 2

    normalized_data = {}
    for k, list_lms in landmarks.items():
        normalized_data[k] = [
            Landmark(x=(lm.x - min_x) * scale + offset_x, 
                     y=(lm.y - min_y) * scale + offset_y, 
                     landmark_id=lm.landmark_id) 
            for lm in list_lms
        ]
    return normalized_data

In [10]:
l = load_pose_from_csv('../datos/MultiViewVisibleImagesHPE_CSV/test/00_15/00_15_1680259460105_rgb.csv')  # Ejemplo de uso

In [12]:
normalized_lms = normalize_landmarks(l, tam_target=100, margin=5)

In [13]:
img_result = plot_landmarks(normalized_lms, tam=100, is_normalized=True)

In [17]:
cv2.imwrite(".borrar.png", img_result)

True